# Azure ML OneLake Datastore Setup

Creates an Azure ML OneLake datastore pointing to a Fabric lakehouse.

In [ ]:
!sudo ACCEPT_EULA=Y apt-get install -y msodbcsql18

In [ ]:
%pip install -q azure-ai-ml azure-identity

In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import OneLakeArtifact, OneLakeDatastore
from azure.core.exceptions import HttpResponseError
from azure.identity import DefaultAzureCredential

# --- Configuration ---
SUBSCRIPTION_ID = "<your-subscription-id>"  # Azure subscription ID
RESOURCE_GROUP = "<your-resource-group>"      # Resource group containing your AML workspace
WORKSPACE_NAME = "<your-aml-workspace>"        # Azure ML workspace name

FABRIC_WORKSPACE_ID = "<your-fabric-workspace-id>"   # Fabric workspace GUID
FABRIC_LAKEHOUSE_ID = "<your-fabric-lakehouse-id>"   # Fabric lakehouse GUID
DATASTORE_NAME_FILES = "fabric_onelake_files"
DATASTORE_NAME_TABLES = "fabric_onelake_tables"
DATA_PATH = "Files/<your-data-folder>"

# --- Create client ---
credential = DefaultAzureCredential()
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)

def create_onelake_datastore(name, artifact_path, description):
    store = OneLakeDatastore(
        name=name,
        description=description,
        one_lake_workspace_name=FABRIC_WORKSPACE_ID,
        endpoint="onelake.dfs.fabric.microsoft.com",
        artifact=OneLakeArtifact(name=artifact_path, type="lake_house"),
    )
    try:
        ds = ml_client.create_or_update(store)
        print(f"Datastore created: {ds.name}")
    except HttpResponseError as exc:
        if "Immutable" in str(exc):
            ds = ml_client.datastores.get(name)
            print(f"Datastore already exists: {ds.name}")
        else:
            raise
    return ds

# --- Files datastore ---
ds_files = create_onelake_datastore(
    DATASTORE_NAME_FILES,
    f"{FABRIC_LAKEHOUSE_ID}/Files",
    "OneLake datastore pointing to lakehouse Files.",
)
print(f"  URI: azureml://datastores/{DATASTORE_NAME_FILES}/paths/{DATA_PATH}")

# --- Tables datastore ---
ds_tables = create_onelake_datastore(
    DATASTORE_NAME_TABLES,
    f"{FABRIC_LAKEHOUSE_ID}/Tables",
    "OneLake datastore pointing to lakehouse Tables.",
)
print(f"  URI: azureml://datastores/{DATASTORE_NAME_TABLES}/paths/dbo/<your-table>")

In [ ]:
import io
import pandas as pd
from azure.storage.filedatalake import DataLakeServiceClient

account_url = "https://onelake.dfs.fabric.microsoft.com"
service_client = DataLakeServiceClient(account_url, credential=credential)
fs_client = service_client.get_file_system_client(FABRIC_WORKSPACE_ID)

def read_onelake_path(fs_client, path):
    """Read all CSV/Parquet files under a OneLake path into a single DataFrame."""
    paths = list(fs_client.get_paths(path=path))
    dfs = []
    for p in paths:
        if p.name.endswith(".parquet"):
            data = fs_client.get_file_client(p.name).download_file().readall()
            dfs.append(pd.read_parquet(io.BytesIO(data)))
        elif p.name.endswith(".csv"):
            data = fs_client.get_file_client(p.name).download_file()
            dfs.append(pd.read_csv(data))
    if not dfs:
        print(f"  No CSV or Parquet files found under {path}")
        return None
    return pd.concat(dfs, ignore_index=True)

# --- Files path ---
files_path = f"{FABRIC_LAKEHOUSE_ID}/Files/noshow"
print(f"Reading from Files: {files_path}")
df_files = read_onelake_path(fs_client, files_path)
if df_files is not None:
    print(f"  Shape: {df_files.shape}")
    display(df_files.head(10))

# --- Tables path ---
tables_path = f"{FABRIC_LAKEHOUSE_ID}/Tables/dbo/<your-table>"
print(f"\nReading from Tables: {tables_path}")
df_tables = read_onelake_path(fs_client, tables_path)
if df_tables is not None:
    print(f"  Shape: {df_tables.shape}")
    display(df_tables.head(10))

## Connect to SQL Endpoint Directly

### Option A: `pyodbc` (standard ODBC)

Uses `ODBC Driver 18 for SQL Server` with Azure AD token auth. Same TDS protocol as any SQL Server connection — works with **Fabric SQL endpoints, Azure SQL Database, Azure SQL MI, or on-prem SQL Server**.

For Fabric: Fabric Portal > Lakehouse > SQL analytics endpoint > copy the **Server** from Connection strings.

In [ ]:
%pip install -q pyodbc

In [ ]:
import struct
import pyodbc
import pandas as pd
from azure.identity import DefaultAzureCredential

# --- SQL Endpoint Configuration ---
# Works with any SQL Server: Fabric SQL endpoint, Azure SQL DB, Azure SQL MI, or on-prem.
# Examples:
#   Fabric:    "xxx.datawarehouse.fabric.microsoft.com"
#   Azure SQL: "yourserver.database.windows.net"
#   On-prem:   "your-server-hostname" or IP address
SQL_SERVER = "<your-sql-server>"
SQL_DATABASE = "<your-database>"

# --- Acquire Azure AD access token (standard SQL Server token scope) ---
credential = DefaultAzureCredential()
token = credential.get_token("https://database.windows.net/.default")
token_bytes = token.token.encode("utf-16-le")
token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)

# --- Standard ODBC connection (same as connecting to any SQL Server) ---
conn_str = (
    f"Driver={{ODBC Driver 18 for SQL Server}};"
    f"Server={SQL_SERVER};"
    f"Database={SQL_DATABASE};"
    f"Encrypt=yes;"
    f"TrustServerCertificate=no;"
)

SQL_COPT_SS_ACCESS_TOKEN = 1256
conn = pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})
print(f"Connected to {SQL_SERVER} / {SQL_DATABASE}")

# --- Query like any SQL Server ---
df_sql = pd.read_sql("SELECT TOP 10 * FROM dbo.<your-table>", conn)
print(f"Shape: {df_sql.shape}")

display(df_sql)conn.close()


## Option B: Connect via `mssql-python` (Microsoft's new Python driver)

No ODBC Driver Manager needed — `mssql-python` uses Direct Database Connectivity (DDBC) under the hood. Supports Entra ID auth natively. See [github.com/microsoft/mssql-python](https://github.com/microsoft/mssql-python).

In [ ]:
%pip install -q mssql-python

In [ ]:
import mssql_python
import pandas as pd

# --- SQL Endpoint Configuration ---
# Same server/database as the pyodbc example above.
# Works with Fabric SQL endpoint, Azure SQL DB, Azure SQL MI, or on-prem SQL Server.
SQL_SERVER = "<your-sql-server>"
SQL_DATABASE = "<your-database>"

# ActiveDirectoryDefault uses the same credential chain as DefaultAzureCredential
# (managed identity on AML compute, Azure CLI locally, etc.)
connection_string = (
    f"SERVER=tcp:{SQL_SERVER},1433;"
    f"DATABASE={SQL_DATABASE};"
    f"Authentication=ActiveDirectoryDefault;"
    f"Encrypt=yes;"
)

conn = mssql_python.connect(connection_string)
print(f"Connected via mssql-python to {SQL_SERVER}")

cursor = conn.cursor()
cursor.execute("SELECT TOP 10 * FROM dbo.<your-table>")
rows = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]

df_mssql = pd.DataFrame(rows, columns=columns)
print(f"Shape: {df_mssql.shape}")
display(df_mssql)

conn.close()

## Option C: Register as an Azure ML Connection

Register the SQL endpoint as an **external connection** in your AML workspace so it can be referenced in pipelines, jobs, and other AML components. See [AML external connections docs](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-connection?view=azureml-api-2).

In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import WorkspaceConnection
from azure.ai.ml.entities._credentials import UsernamePasswordConfiguration
from azure.identity import DefaultAzureCredential

# Reuse existing config
SUBSCRIPTION_ID = "<your-subscription-id>"
RESOURCE_GROUP = "<your-resource-group>"
WORKSPACE_NAME = "<your-aml-workspace>"

SQL_SERVER = "<your-sql-server>"
SQL_DATABASE = "<your-database>"
SQL_PORT = 1433

credential = DefaultAzureCredential()
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)

# Create an Azure SQL DB type connection in AML
# Note: For Entra-ID-only endpoints you may leave username/password as placeholders
# and use the connection as a reference; actual auth happens at runtime.
target = f"Server=tcp:{SQL_SERVER},{SQL_PORT};Database={SQL_DATABASE};Trusted_Connection=False;Encrypt=True;Connection Timeout=30"

ws_connection = WorkspaceConnection(
    name="ml-sql-endpoint",
    type="azure_sql_db",
    target=target,
    credentials=UsernamePasswordConfiguration(
        username="placeholder",
        password="placeholder",
    ),
)

connection = ml_client.connections.create_or_update(ws_connection)
print(f"AML Connection created: {connection.name}")
print(f"  Type: {connection.type}")
print(f"  Target: {connection.target}")